## Hyper-parameter Search

Every scikit-learn model ships with default settings — but those defaults are rarely optimal for *your* data. **How do you find the right knobs to turn without peeking at the test set?**

In this lesson we learn to:

- Distinguish **parameters** (learned during `.fit`) from **hyper-parameters** (set at construction time).
- Recognize the **five elements** of hyper-parameter search in scikit-learn.
- Explain the role of each element in the search workflow.
- Compare **Grid Search**, **Random Search**, and **Bayesian Search**.
- Recognize more efficient, **model-specific cross-validation** helpers (`RidgeCV`, `LogisticRegressionCV`).

## Hyper-parameters vs. parameters

- **Parameters** are what models learn during `.fit(X, y)`.
- **Hyper-parameters** are passed to the constructor of the estimator (or transformer).

To inspect all names and current values, call `estimator.get_params()`:

In [1]:
# https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.get_params()

{'copy_X': True,
 'fit_intercept': True,
 'n_jobs': None,
 'positive': False,
 'tol': 1e-06}

In [2]:
# https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()
model.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [3]:
# https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.get_params()

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [4]:
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor()
model.get_params()

{'categorical_features': 'from_dtype',
 'early_stopping': 'auto',
 'interaction_cst': None,
 'l2_regularization': 0.0,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_bins': 255,
 'max_depth': None,
 'max_features': 1.0,
 'max_iter': 100,
 'max_leaf_nodes': 31,
 'min_samples_leaf': 20,
 'monotonic_cst': None,
 'n_iter_no_change': 10,
 'quantile': None,
 'random_state': None,
 'scoring': 'loss',
 'tol': 1e-07,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [5]:
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor()
model.get_params()

{'categorical_features': 'from_dtype',
 'early_stopping': 'auto',
 'interaction_cst': None,
 'l2_regularization': 0.0,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_bins': 255,
 'max_depth': None,
 'max_features': 1.0,
 'max_iter': 100,
 'max_leaf_nodes': 31,
 'min_samples_leaf': 20,
 'monotonic_cst': None,
 'n_iter_no_change': 10,
 'quantile': None,
 'random_state': None,
 'scoring': 'loss',
 'tol': 1e-07,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

### Transformer hyper-parameters

In scikit-learn, **transformers are estimators too** and have their own hyper-parameters.

For example, [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) has a `degree` parameter:

In [6]:
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures()
poly.get_params()

{'degree': 2, 'include_bias': True, 'interaction_only': False, 'order': 'C'}

Likewise, [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) has a `strategy` hyper-parameter (`"mean"`, `"median"`, …):

In [7]:
# https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(
    strategy="mean",  # or "median", "most_frequent", "constant"
)
imputer.get_params()

{'add_indicator': False,
 'copy': True,
 'fill_value': None,
 'keep_empty_features': False,
 'missing_values': nan,
 'strategy': 'mean'}

## Hyper-parameter tuning methods

It is possible — and **recommended** — to search the hyper-parameter space for the best [**cross-validation**](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation) score.

The five elements of HP search are:

1. an **estimator**
2. a **parameter space**
3. a **search method** (tuning algorithm)
4. a **cross-validation scheme**
5. a [**scoring function**](https://scikit-learn.org/stable/modules/grid_search.html#gridsearch-scoring) (e.g. `accuracy_score` for classification)

### Search method and parameter space

1. [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) exhaustively tries every combination in the grid.
2. [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) samples a fixed number of candidates from a distribution over the space.
3. While Grid and Random search are "blind" strategies, [`skopt.BayesSearchCV`](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html) is "informed": it builds a **surrogate model** that updates its sampling distribution based on previous trials. On some datasets it reaches the same score with **7× fewer iterations and 5× less time**.

![Grid vs Random vs Bayesian search](../assets/hp_search_methods.png)

Fourth method based on genetic algorithms: [Population based training | Google DeepMind](https://deepmind.google/blog/population-based-training-of-neural-networks/).

### Efficient model-specific cross-validation

Some models can fit data for a **range** of parameter values almost as efficiently as for a single value. scikit-learn exposes dedicated helpers:

- `RidgeCV` for regression
- `LogisticRegressionCV` for classification

See: [Model-specific cross-validation](https://scikit-learn.org/stable/modules/grid_search.html#model-specific-cross-validation)

## HP tuning workflow

```{mermaid}
graph TD
    classDef purpleNode fill:#ccccff,stroke:#000,stroke-width:1px,color:#000;

    Parameters(param_grid):::purpleNode
    Dataset(data):::purpleNode
    CV(Search):::purpleNode
    Trainingdata(train_data):::purpleNode
    Testdata(test_data):::purpleNode
    Retrainedmodel(search.best_estimator_):::purpleNode
    Finalevaluation(".score()"):::purpleNode

    Dataset --> Trainingdata
    Dataset --> Testdata
    Parameters --> CV
    Trainingdata --> CV
    Trainingdata --> Retrainedmodel
    CV --> Retrainedmodel
    Retrainedmodel --> Finalevaluation
    Testdata --> Finalevaluation
```

First:

- Split `data` into `train_data` and `test_data`.
- Specify the search space `param_grid`.

Then:

1. The search algorithm (e.g. `GridSearchCV`) iterates over train/validation splits; for each candidate it trains and scores.
2. `search.best_params_` initializes a new model, refit on the **entire** training set → `search.best_estimator_`.
3. Evaluate the refit model on the **held-out test set**.

![Inner cross-validation during HP search](../assets/inner_cross_validation.png)

### Example: Bayesian search (cross-validated)

We search the parameter space of a gradient-boosting model using [`skopt.BayesSearchCV`](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html):

- It shares the same API as `GridSearchCV`/`RandomizedSearchCV` (`fit`, `best_params_`, `best_estimator_`, `score`, …).
- Instead of a grid or distributions, we pass a `search_spaces` dict mapping each parameter to a [skopt dimension](https://scikit-optimize.github.io/stable/modules/classes.html#module-skopt.space.space): [`Integer`](https://scikit-optimize.github.io/stable/modules/generated/skopt.space.space.Integer.html), [`Real`](https://scikit-optimize.github.io/stable/modules/generated/skopt.space.space.Real.html), or [`Categorical`](https://scikit-optimize.github.io/stable/modules/generated/skopt.space.space.Categorical.html).
- `n_iter` controls how many candidates are evaluated; each one is chosen by a **surrogate model** based on the scores seen so far.

In [8]:
# https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_regression.html
from sklearn.datasets import make_regression

# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
from sklearn.ensemble import HistGradientBoostingRegressor

# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
from sklearn.model_selection import train_test_split

# https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html
from skopt import BayesSearchCV

# https://scikit-optimize.github.io/stable/modules/classes.html#module-skopt.space.space
from skopt.space import Integer, Real

In [9]:
X, y = make_regression(
    n_samples=20640,
    n_features=8,
    noise=0.1,
    random_state=0,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=0,
)

In [10]:
search_spaces = {
    "max_depth": Integer(5, 9),                           # tree depth in {5, …, 9}
    "learning_rate": Real(0.01, 0.2, prior="log-uniform"),  # step size, searched on a log scale
    "max_iter": Integer(50, 150),                         # number of boosting iterations
}

search = BayesSearchCV(
    estimator=HistGradientBoostingRegressor(random_state=0),
    search_spaces=search_spaces,
    n_iter=5,           # evaluate 5 candidates guided by the surrogate model
    random_state=0,     # reproducible sampling
)

search.fit(X_train, y_train)

,estimator,HistGradientB...andom_state=0)
,search_spaces,"{'learning_rate': Real(low=0.01...m='normalize'), 'max_depth': Integer(low=5...m='normalize'), 'max_iter': Integer(low=5...m='normalize')}"
,n_iter,5
,random_state,0
,optimizer_kwargs,None
,scoring,None
,fit_params,None
,n_jobs,1
,n_points,1
,iid,'deprecated'
,refit,True


In [11]:
best_model = search.best_estimator_
best_model

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.04906822016229648
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",112
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",8
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [12]:
best_model.score(X_test, y_test)

0.9705105943200796

## Pipelines are also estimators

Every search — [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html), and [`skopt.BayesSearchCV`](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html) — can tune parameters of **nested** estimators such as [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) or [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) using the `<step>__<parameter>` syntax:

In [13]:
# https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
from sklearn.pipeline import Pipeline

# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html
from sklearn.ensemble import HistGradientBoostingClassifier

# https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html
from sklearn.datasets import make_moons

# https://scikit-optimize.github.io/stable/modules/classes.html#module-skopt.space.space
from skopt.space import Categorical

In [14]:
X, y = make_moons(
    n_samples=500,
    noise=0.3,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=0,
)

In [15]:
pipe = Pipeline([
    ("imputer", SimpleImputer()),
    ("model", HistGradientBoostingClassifier()),
])

# Syntax: <step_name>__<parameter_name>
search_spaces = {
    "imputer__strategy": Categorical(["mean", "median"]),  # discrete choices
    "model__max_iter": Integer(100, 200),                  # range of boosting iterations
    "model__max_depth": Categorical([3, 5, None]),         # None means unlimited depth
}

search = BayesSearchCV(
    estimator=pipe,
    search_spaces=search_spaces,
    n_iter=8,           # evaluate 8 candidates guided by the surrogate model
    cv=5,               # 5-fold cross-validation inside the search
    random_state=0,     # reproducible sampling
)

search.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,search_spaces,"{'imputer__strategy': Categorical(c...), prior=None), 'model__max_depth': Categorical(c...), prior=None), 'model__max_iter': Integer(low=1...m='normalize')}"
,n_iter,8
,cv,5
,random_state,0
,optimizer_kwargs,None
,scoring,None
,fit_params,None
,n_jobs,1
,n_points,1
,iid,'deprecated'


In [16]:
print(f"Best settings found: {search.best_params_}")
print(f"CV accuracy of best model: {search.best_score_:.2%}")

Best settings found: OrderedDict({'imputer__strategy': 'median', 'model__max_depth': 3, 'model__max_iter': 134})
CV accuracy of best model: 91.20%


In [17]:
best_pipeline = search.best_estimator_
best_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the foll

In [18]:
best_pipeline.score(X_test, y_test)

0.888

Refer to [Pipeline: chaining estimators](https://scikit-learn.org/stable/modules/compose.html#pipeline) for more details.